# ISOM5240 Fine-tuning Notebook — Pipeline 1: Shelf-life Classification

**Workflow:**
- Phase 1 → Compare 3 pre-trained models via keyword mapping (NO training) → select best
- Phase 2 → Fine-tune only the selected model on full dataset
- Phase 3 → Evaluate + export Excel + push to HuggingFace Hub

**Datasets:**
- Grocery Store Dataset (GitHub) → short_shelf + medium_shelf
- Kaggle Household Products → non_perishable

**Task:** 3-class image classification (short_shelf / medium_shelf / non_perishable)

## Step 1: Install dependencies

In [ ]:
!pip install transformers datasets evaluate accelerate pillow scikit-learn -q
!pip install huggingface_hub -q

## Step 2: GPU check

In [ ]:
import torch
import os
import time
import numpy as np
import glob
from PIL import Image

if not torch.cuda.is_available():
    raise RuntimeError("No GPU! Runtime → Change runtime type → T4 GPU → Save, then Run All.")

print(f"Using GPU: {torch.cuda.get_device_name(0)}")

## Step 3: Login to HuggingFace + setup Kaggle

In [ ]:
from huggingface_hub import login
from google.colab import userdata

login(token=userdata.get('HF_TOKEN'))
os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')
os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')
print("HuggingFace + Kaggle ready")

## Step 4: Download both datasets

In [ ]:
!git clone https://github.com/marcusklasson/GroceryStoreDataset.git
!kaggle datasets download -d taru149/householdproducts
!unzip -q householdproducts.zip -d household/
print("Both datasets downloaded!")

## Step 5: Build 3-class dataset

- short_shelf (0): fruits, vegetables
- medium_shelf (1): dairy, juice, milk
- non_perishable (2): household products

In [ ]:
SHORT_KEYWORDS = [
    "apple", "avocado", "banana", "kiwi", "lemon", "lime", "mango",
    "melon", "nectarine", "orange", "papaya", "passion", "peach",
    "pear", "pineapple", "plum", "pomegranate", "grapefruit",
    "satsuma", "watermelon", "asparagus", "aubergine", "cabbage",
    "carrot", "cucumber", "garlic", "ginger", "leek", "mushroom",
    "onion", "pepper", "potato", "red-beet", "tomato", "zucchini"
]

MEDIUM_KEYWORDS = [
    "juice", "milk", "oat", "sour-cream", "sour-milk",
    "soy", "yoghurt", "cream"
]

LABEL_NAMES = ["short_shelf", "medium_shelf", "non_perishable"]

def collect_grocery_images(split_dir):
    images, labels = [], []
    for class_dir in sorted(os.listdir(split_dir)):
        class_path = os.path.join(split_dir, class_dir)
        if not os.path.isdir(class_path):
            continue
        name = class_dir.lower()
        if any(kw in name for kw in SHORT_KEYWORDS):
            label = 0
        elif any(kw in name for kw in MEDIUM_KEYWORDS):
            label = 1
        else:
            continue
        for f in os.listdir(class_path):
            if f.lower().endswith((".jpg", ".jpeg", ".png")):
                images.append(os.path.join(class_path, f))
                labels.append(label)
    return images, labels

grocery_train_imgs, grocery_train_labels = collect_grocery_images("GroceryStoreDataset/dataset/train")
grocery_test_imgs, grocery_test_labels = collect_grocery_images("GroceryStoreDataset/dataset/test")

household_imgs = (glob.glob("household/**/*.jpg", recursive=True) +
                  glob.glob("household/**/*.png", recursive=True) +
                  glob.glob("household/**/*.jpeg", recursive=True))

print(f"Grocery train: {len(grocery_train_imgs)} | Grocery test: {len(grocery_test_imgs)}")
print(f"Household: {len(household_imgs)}")

In [ ]:
from sklearn.model_selection import train_test_split

if len(household_imgs) > 0:
    hh_train, hh_test = train_test_split(household_imgs, test_size=0.2, random_state=42)
else:
    hh_train, hh_test = [], []

all_train_paths = grocery_train_imgs + hh_train
all_train_labels = grocery_train_labels + [2] * len(hh_train)

all_test_paths = grocery_test_imgs + hh_test
all_test_labels = grocery_test_labels + [2] * len(hh_test)

all_train_labels_np = np.array(all_train_labels)
all_test_labels_np = np.array(all_test_labels)

print(f"Combined train: {len(all_train_paths)} | Combined test: {len(all_test_paths)}")
for i, name in enumerate(LABEL_NAMES):
    print(f"  {name}: train={int((all_train_labels_np==i).sum())}, test={int((all_test_labels_np==i).sum())}")

---
# PHASE 1: Model Selection (pre-trained + keyword mapping, NO training)

Compare 3 models using ImageNet labels mapped to shelf-life categories via keywords.
Same approach as the Pipeline 1 comparison notebook.

## Step 6: Define candidate models and keyword mapping

In [ ]:
CANDIDATE_MODELS = {
    "ViT-base": "google/vit-base-patch16-224",
    "ResNet-50": "microsoft/resnet-50",
    "Swin-tiny": "microsoft/swin-tiny-patch4-window7-224",
}

# ImageNet label → shelf-life category
IMAGENET_SHORT = [
    "banana", "orange", "strawberry", "apple", "lemon", "pineapple",
    "pomegranate", "fig", "jackfruit", "mango", "broccoli", "cucumber",
    "mushroom", "meat", "egg", "bakery", "bread", "grocery", "fruit",
    "vegetable", "food", "pizza", "hotdog", "pretzel", "bagel", "dough",
    "zucchini", "pepper", "cauliflower", "artichoke", "potato", "cabbage",
    "head cabbage", "corn", "acorn squash", "spaghetti squash",
    "butternut squash", "ice cream", "custard apple"
]

IMAGENET_MEDIUM = [
    "bottle", "can", "jar", "packet", "carton", "sauce", "wine",
    "beer", "juice", "water", "pop", "cup", "coffee", "espresso",
    "milk can", "water bottle", "wine bottle", "beer bottle", "pop bottle"
]

def map_to_shelf_life(imagenet_label):
    label = imagenet_label.lower()
    if any(kw in label for kw in IMAGENET_SHORT):
        return 0
    elif any(kw in label for kw in IMAGENET_MEDIUM):
        return 1
    else:
        return 2  # default: non_perishable

print("Candidates and keyword mapping ready")

## Step 7: Compare 3 models (pre-trained, no training)

In [ ]:
from transformers import pipeline as hf_pipeline
import pandas as pd

def evaluate_pretrained_p1(model_key, model_path, test_images, test_labels):
    print(f"\nEvaluating: {model_key}")

    pipe = hf_pipeline("image-classification", model=model_path, device=0)
    total_params = sum(p.numel() for p in pipe.model.parameters())

    correct = 0
    total = len(test_images)
    inference_times = []

    for i in range(total):
        img = Image.open(test_images[i]).convert("RGB")
        t0 = time.time()
        result = pipe(img, top_k=1)
        inference_times.append(time.time() - t0)

        pred = map_to_shelf_life(result[0]["label"])
        if pred == test_labels[i]:
            correct += 1

    accuracy = correct / total
    avg_ms = np.mean(inference_times) * 1000

    print(f"  Accuracy: {accuracy:.4f} | Speed: {avg_ms:.1f}ms | Params: {total_params/1e6:.1f}M")

    return {
        "Model": model_key,
        "Parameters (M)": f"{total_params/1e6:.1f}M",
        "Accuracy (pre-trained)": round(accuracy, 4),
        "Avg Inference (ms)": round(avg_ms, 1),
        "Test Samples": total,
    }

pretrained_results = []
for key, path in CANDIDATE_MODELS.items():
    r = evaluate_pretrained_p1(key, path, all_test_paths, all_test_labels)
    pretrained_results.append(r)

df_selection = pd.DataFrame(pretrained_results)
print("\n" + "="*60)
print("PHASE 1 RESULTS: Pre-trained Model Comparison")
print("="*60)
print(df_selection.to_string(index=False))

## Step 8: Select best model

In [ ]:
best = max(pretrained_results, key=lambda x: x["Accuracy (pre-trained)"])
SELECTED_MODEL_KEY = best["Model"]
SELECTED_MODEL_PATH = CANDIDATE_MODELS[SELECTED_MODEL_KEY]

print(f"Selected: {SELECTED_MODEL_KEY}")
print(f"  Pre-trained accuracy: {best['Accuracy (pre-trained)']}")
print(f"  Inference speed: {best['Avg Inference (ms)']}ms")

---
# PHASE 2: Fine-tune the selected model

Now fine-tune the best model so it directly outputs short_shelf / medium_shelf / non_perishable,
instead of relying on keyword mapping.

## Step 9: Build HuggingFace Dataset

In [ ]:
from datasets import Dataset, DatasetDict

def load_image(path):
    return Image.open(path).convert("RGB")

print("Loading train images...")
train_dataset = Dataset.from_dict({
    "image": [load_image(p) for p in all_train_paths],
    "label": all_train_labels,
})

print("Loading test images...")
test_dataset = Dataset.from_dict({
    "image": [load_image(p) for p in all_test_paths],
    "label": all_test_labels,
})

split = train_dataset.train_test_split(test_size=0.1, seed=42)
dataset = DatasetDict({
    "train": split["train"],
    "validation": split["test"],
    "test": test_dataset,
})

print(f"Train:      {len(dataset['train'])}")
print(f"Validation: {len(dataset['validation'])}")
print(f"Test:       {len(dataset['test'])}")

## Step 10: Prepare data for selected model

In [ ]:
from transformers import AutoImageProcessor

processor = AutoImageProcessor.from_pretrained(SELECTED_MODEL_PATH)

def preprocess(batch):
    images = [img.convert("RGB") for img in batch["image"]]
    inputs = processor(images=images, return_tensors="pt")
    inputs["label"] = batch["label"]
    return inputs

dataset["train"].set_transform(preprocess)
dataset["validation"].set_transform(preprocess)
dataset["test"].set_transform(preprocess)

print(f"Transform set for {SELECTED_MODEL_KEY}")

## Step 11: Load model (3-class head)

In [ ]:
from transformers import AutoModelForImageClassification

model = AutoModelForImageClassification.from_pretrained(
    SELECTED_MODEL_PATH,
    num_labels=len(LABEL_NAMES),
    id2label={i: l for i, l in enumerate(LABEL_NAMES)},
    label2id={l: i for i, l in enumerate(LABEL_NAMES)},
    ignore_mismatched_sizes=True,
)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model: {SELECTED_MODEL_KEY} | Params: {total_params:,}")

## Step 12: Train

In [ ]:
from transformers import TrainingArguments, Trainer
import evaluate

accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

training_args = TrainingArguments(
    output_dir=f"./shelf-life-{SELECTED_MODEL_KEY.lower()}",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_steps=50,
    remove_unused_columns=False,
    fp16=True,
    dataloader_num_workers=2,
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    compute_metrics=compute_metrics,
)

train_start = time.time()
trainer.train()
train_time = time.time() - train_start
print(f"\nTraining complete in {train_time/60:.1f} minutes")

## Step 13: Evaluate — Accuracy

In [ ]:
ft_results = trainer.evaluate(dataset["test"])
print(f"Test Accuracy: {ft_results['eval_accuracy']:.4f}")
print(f"Test Loss:     {ft_results['eval_loss']:.4f}")

## Step 14: Evaluate — Precision, Recall, F1, Confusion Matrix

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

pipe_ft = hf_pipeline("image-classification", model=model, image_processor=processor, device=0)

y_true = []
y_pred = []

for i, path in enumerate(all_test_paths):
    img = Image.open(path).convert("RGB")
    true_label = LABEL_NAMES[all_test_labels[i]]
    pred = pipe_ft(img, top_k=1)
    y_true.append(true_label)
    y_pred.append(pred[0]["label"])

print("Classification Report:")
print(classification_report(y_true, y_pred, digits=4))
print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred, labels=LABEL_NAMES))

## Step 15: Inference speed

In [ ]:
inf_times = []
for path in all_test_paths[:100]:
    img = Image.open(path).convert("RGB")
    t0 = time.time()
    pipe_ft(img)
    inf_times.append(time.time() - t0)

ft_avg_ms = np.mean(inf_times) * 1000
print(f"Avg inference: {ft_avg_ms:.1f}ms per image")

## Step 16: Before vs After comparison

In [ ]:
selected_phase1 = next(r for r in pretrained_results if r["Model"] == SELECTED_MODEL_KEY)

comparison = pd.DataFrame([
    {
        "Stage": "Pre-trained + keyword mapping",
        "Model": SELECTED_MODEL_KEY,
        "Accuracy": selected_phase1["Accuracy (pre-trained)"],
        "Avg Inference (ms)": selected_phase1["Avg Inference (ms)"],
    },
    {
        "Stage": "Fine-tuned (3-class direct)",
        "Model": SELECTED_MODEL_KEY,
        "Accuracy": round(ft_results["eval_accuracy"], 4),
        "Avg Inference (ms)": round(ft_avg_ms, 1),
    },
])

print("="*60)
print("Before vs After Fine-tuning")
print("="*60)
print(comparison.to_string(index=False))

## Step 17: Export Excel

In [ ]:
with pd.ExcelWriter("P1_Experimental_results.xlsx") as writer:
    df_selection.to_excel(writer, sheet_name="P1 Model Selection", index=False)
    comparison.to_excel(writer, sheet_name="P1 Fine-tune Result", index=False)

print("Saved!")
from google.colab import files
files.download("P1_Experimental_results.xlsx")

## Step 18: Push to HuggingFace Hub

In [ ]:
HUB_MODEL_ID = "Alisa-Sun/shelf-life-classification"  # TODO: Change if needed

model.push_to_hub(HUB_MODEL_ID, commit_message=f"Fine-tuned {SELECTED_MODEL_KEY} | acc={ft_results['eval_accuracy']:.4f}")
processor.push_to_hub(HUB_MODEL_ID)

print(f"Model pushed to: https://huggingface.co/{HUB_MODEL_ID}")

## Step 19: Final inference test

In [ ]:
pipe_final = hf_pipeline("image-classification", model=HUB_MODEL_ID)

print("Testing 5 images:")
for i in range(min(5, len(all_test_paths))):
    img = Image.open(all_test_paths[i]).convert("RGB")
    true_label = LABEL_NAMES[all_test_labels[i]]
    pred = pipe_final(img)
    status = "✅" if pred[0]["label"] == true_label else "❌"
    print(f"  {status} True: {true_label} | Pred: {pred[0]['label']} ({pred[0]['score']:.3f})")

---
## Summary

| Step | Content |
|------|---------|
| 1-5 | Setup + download Grocery Store + Household Products → build 3-class dataset |
| 6-8 | **Phase 1:** Pre-trained + keyword mapping comparison (no training) → select best |
| 9-12 | **Phase 2:** Fine-tune selected model (3 epochs, full data) |
| 13-15 | Evaluate: accuracy, precision, recall, confusion matrix, speed |
| 16 | Before (keyword mapping) vs After (fine-tuned) comparison |
| 17 | Export Excel |
| 18-19 | Push to Hub + final test |

**Key difference from Pipeline 2:** Phase 1 here uses keyword mapping to compare models (no training needed), while Pipeline 2 used quick 1-epoch fine-tuning. Both approaches are valid for model selection.